# Feature Engineering & Training Export

**Purpose:** Build a modeling-ready customer × month table: locked `Is_Churned` (ADR 004) plus **month-`t`-only** features, then export `training_data.csv`.

**Depends on:** churn rules locked in `01_data_integrity_and_eda.ipynb` / [ADR 004](../../Docs/Architecture/ADRs/004-churn-target-and-predictive-window.md).

**Anti-leakage:** every feature uses data available through the end of observation month `t`. The label is the churn state in month `t+1` only.

**Inputs:** `StartingFiles/customers_tm1_e.csv`, `StartingFiles/transactions_tm1_e.csv`  
**Output:** `Data/processed/training_data.csv` (eligible rows only; `*.csv` is gitignored — regenerate by running this notebook)

**Column names:** snake_case only — see `.cursor/skills/validate-csv-schema/schema.json`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook lives in Code/Sprint1/ — resolve paths from repo root regardless of kernel cwd.
REPO_ROOT = Path('../..').resolve()
DATA_DIR = REPO_ROOT / 'StartingFiles'
OUT_DIR = REPO_ROOT / 'Data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'training_data.csv'

assert (DATA_DIR / 'customers_tm1_e.csv').exists(), f'Missing customers CSV under {DATA_DIR}'
assert (DATA_DIR / 'transactions_tm1_e.csv').exists(), f'Missing transactions CSV under {DATA_DIR}'

# Locked constants (same as EDA notebook / ADR 004).
GRACE_MONTHS = 2
ROLL_WINDOW = 3  # months of history for rolling behavioural features


## 1. Load raw CSVs

Same sources as the EDA notebook. Transaction file is large (~350 MB on disk).


In [ ]:
customers_df = pd.read_csv(DATA_DIR / 'customers_tm1_e.csv')
transactions_df = pd.read_csv(DATA_DIR / 'transactions_tm1_e.csv')

print(f'customers={len(customers_df):,}  transactions={len(transactions_df):,}')
print('customer columns:', list(customers_df.columns))
print('transaction columns:', list(transactions_df.columns))


## 2. Monthly panel + locked `Is_Churned`

Rebuild the customer × month grain used in EDA:

1. Flag `$0` placeholders → activity = nonzero cash movement only
2. Aggregate to `(customer_id, Month)`
3. `Running_Balance = start_balance.fillna(0) + cumsum(monthly net)` (Sam)
4. Label: at month `t`, `Is_Churned` = whether month `t+1` is rounded-£0 **and** inactive
5. `Is_Train_Eligible` = tenure ≥ `GRACE_MONTHS` and next month known


In [ ]:
# --- Dates & placeholder activity flag ---
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])
transactions_df['Month'] = transactions_df['transaction_date'].dt.to_period('M')

customers_df['creation_date'] = pd.to_datetime(customers_df['creation_date'])
customers_df['dob'] = pd.to_datetime(customers_df['dob'])

zero_mask = (
    (transactions_df['amount'] == 0)
    & (transactions_df['deposit'] == 0)
    & (transactions_df['withdrawal'] == 0)
)
transactions_df['is_active_tx'] = ~zero_mask

# --- Monthly rollup (cash-flow features at month t) ---
monthly_tx = transactions_df.groupby(['customer_id', 'Month'], as_index=False).agg(
    Monthly_Transaction_Count=('amount', 'size'),
    Monthly_Active_Txn_Count=('is_active_tx', 'sum'),
    Monthly_Amount_Sum=('amount', 'sum'),
    Monthly_Deposit_Sum=('deposit', 'sum'),
    Monthly_Withdrawal_Sum=('withdrawal', 'sum'),
)

cust_cols = customers_df[
    ['customer_id', 'start_balance', 'creation_date', 'dob', 'state']
].copy()
# Stakeholder decision (Sam): treat null opening balances as $0.
cust_cols['start_balance'] = cust_cols['start_balance'].fillna(0)
monthly_tx = monthly_tx.merge(cust_cols, on='customer_id', how='left')

monthly_tx = monthly_tx.sort_values(['customer_id', 'Month']).reset_index(drop=True)
monthly_tx['Running_Balance'] = (
    monthly_tx['start_balance']
    + monthly_tx.groupby('customer_id')['Monthly_Amount_Sum'].cumsum()
)
monthly_tx['Running_Balance_Rounded'] = monthly_tx['Running_Balance'].round(0)

month_end = monthly_tx['Month'].dt.to_timestamp(how='end')
monthly_tx['Months_Since_Creation'] = (
    (month_end.dt.year - monthly_tx['creation_date'].dt.year) * 12
    + (month_end.dt.month - monthly_tx['creation_date'].dt.month)
)

# Locked "gone this calendar month" state (used for label shift, not as a future peek).
monthly_tx['gone_this_month'] = (
    (monthly_tx['Running_Balance_Rounded'] == 0)
    & (monthly_tx['Monthly_Active_Txn_Count'] == 0)
).astype(int)

# Label for row at month t = whether month t+1 is "gone".
monthly_tx['Is_Churned'] = (
    monthly_tx.groupby('customer_id')['gone_this_month'].shift(-1)
)
monthly_tx['Is_Train_Eligible'] = (
    (monthly_tx['Months_Since_Creation'] >= GRACE_MONTHS)
    & monthly_tx['Is_Churned'].notna()
)

print(
    f'monthly rows={len(monthly_tx):,}  '
    f'eligible={monthly_tx["Is_Train_Eligible"].sum():,}  '
    f'pos_rate={monthly_tx.loc[monthly_tx["Is_Train_Eligible"], "Is_Churned"].mean():.4%}'
)


## 3. Feature engineering (observation month `t` only)

| Group | Features | Why |
|-------|----------|-----|
| Customer | `age_years`, `state`, `start_balance`, `Months_Since_Creation` | Demographics / tenure Sam left open for us to choose |
| Level | `Running_Balance`, rounded balance, monthly counts & cash sums | Current account state at end of `t` |
| Dynamics | MoM balance change, lags, rolling means | Balance / activity trajectory without using `t+1` |
| Dormancy | inactive flag, consecutive inactive streak, recent inactive share | Separates quiet-but-funded from exit-shaped behaviour |
| Current exit shape | `gone_this_month` | Known at end of `t`; strong but **not** leakage (label is still `t+1`) |

**Not included as features:** anything from month `t+1`, and raw `Is_Churned` (that is the target).


In [ ]:
g = monthly_tx.groupby('customer_id', sort=False)

# --- Customer: age at month-end ---
monthly_tx['age_years'] = (
    (month_end - monthly_tx['dob']).dt.days / 365.25
).astype('float32')

# --- Dynamics: lags are prior months only (no t+1) ---
monthly_tx['Balance_MoM_Change'] = monthly_tx['Running_Balance'] - g['Running_Balance'].shift(1)
monthly_tx['Active_Txn_Lag1'] = g['Monthly_Active_Txn_Count'].shift(1)
monthly_tx['Active_Txn_Lag2'] = g['Monthly_Active_Txn_Count'].shift(2)
monthly_tx['Active_Txn_Lag3'] = g['Monthly_Active_Txn_Count'].shift(3)
monthly_tx['Amount_Lag1'] = g['Monthly_Amount_Sum'].shift(1)

# Rolling means of *prior* months: shift first, then rolling within customer.
def prior_roll_mean(col: str) -> pd.Series:
    shifted = g[col].shift(1)
    return shifted.groupby(monthly_tx['customer_id']).transform(
        lambda s: s.rolling(ROLL_WINDOW, min_periods=1).mean()
    )

monthly_tx['Active_Txn_Roll3_Mean'] = prior_roll_mean('Monthly_Active_Txn_Count')
monthly_tx['Amount_Roll3_Mean'] = prior_roll_mean('Monthly_Amount_Sum')
monthly_tx['Balance_Roll3_Mean'] = prior_roll_mean('Running_Balance')

# --- Dormancy ---
monthly_tx['Is_Inactive_Month'] = (monthly_tx['Monthly_Active_Txn_Count'] == 0).astype(int)

# Consecutive inactive months ending at t (vectorized per customer via cumsum reset).
def inactive_streak(s: pd.Series) -> pd.Series:
    # When active (0), start a new group; cumsum within inactive runs = streak length.
    is_inactive = s.to_numpy(dtype=bool)
    reset = ~is_inactive
    group_ids = np.cumsum(reset)
    # Zero-out active months after group cumcount.
    streak = pd.Series(is_inactive.astype(np.int32), index=s.index)
    streak = streak.groupby(group_ids).cumsum()
    streak = streak.where(is_inactive, 0).astype(np.int32)
    return streak

monthly_tx['Consecutive_Inactive_Months'] = g['Is_Inactive_Month'].transform(inactive_streak)
monthly_tx['Inactive_Share_Prior3'] = prior_roll_mean('Is_Inactive_Month')

# Deposit vs withdrawal intensity in month t.
withdraw_abs = monthly_tx['Monthly_Withdrawal_Sum'].abs()
flow_denom = monthly_tx['Monthly_Deposit_Sum'] + withdraw_abs
monthly_tx['Deposit_Share_Of_Flow'] = np.where(
    flow_denom > 0,
    monthly_tx['Monthly_Deposit_Sum'] / flow_denom,
    0.0,
)

print('Feature columns added. Sample:')
display(
    monthly_tx[
        [
            'customer_id',
            'Month',
            'Months_Since_Creation',
            'age_years',
            'Running_Balance',
            'Monthly_Active_Txn_Count',
            'Balance_MoM_Change',
            'Consecutive_Inactive_Months',
            'gone_this_month',
            'Is_Churned',
        ]
    ].head(8)
)


## 4. Assemble training frame & export

Keep only `Is_Train_Eligible` rows. Drop helper / PII-ish date columns not needed for XGBoost input (`dob`, `creation_date`). Keep `customer_id` + `Month` for temporal splits and join-backs in Sprint 2.

`Month` is written as `YYYY-MM` string so the CSV round-trips cleanly.


In [ ]:
FEATURE_COLS = [
    # keys / split helpers (not all are model inputs — Sprint 2 will select)
    'customer_id',
    'Month',
    # customer
    'state',
    'age_years',
    'start_balance',
    'Months_Since_Creation',
    # level (month t)
    'Running_Balance',
    'Running_Balance_Rounded',
    'Monthly_Transaction_Count',
    'Monthly_Active_Txn_Count',
    'Monthly_Amount_Sum',
    'Monthly_Deposit_Sum',
    'Monthly_Withdrawal_Sum',
    'Deposit_Share_Of_Flow',
    'Is_Inactive_Month',
    'gone_this_month',
    # dynamics / dormancy
    'Balance_MoM_Change',
    'Active_Txn_Lag1',
    'Active_Txn_Lag2',
    'Active_Txn_Lag3',
    'Amount_Lag1',
    'Active_Txn_Roll3_Mean',
    'Amount_Roll3_Mean',
    'Balance_Roll3_Mean',
    'Consecutive_Inactive_Months',
    'Inactive_Share_Prior3',
]

TARGET_COL = 'Is_Churned'

train_df = monthly_tx.loc[monthly_tx['Is_Train_Eligible'], FEATURE_COLS + [TARGET_COL]].copy()
train_df['Month'] = train_df['Month'].astype(str)
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)

# Early months lack full lag history — leave NaN for the model/imputer in Sprint 2.
n_null_lag = int(train_df['Active_Txn_Lag1'].isna().sum())
print(f'training rows={len(train_df):,}')
print(f'positive rate={train_df[TARGET_COL].mean():.4%}')
print(f'rows with missing Active_Txn_Lag1 (expected early tenure)={n_null_lag:,}')
print(f'columns={len(train_df.columns)}')
display(train_df.head())
display(train_df.describe(include='all').T.head(20))


In [ ]:
# Export cleaned training set for Sprint 2 / MLflow.
train_df.to_csv(OUT_PATH, index=False)
size_mb = OUT_PATH.stat().st_size / (1024 * 1024)
print(f'Wrote {OUT_PATH}')
print(f'File size: {size_mb:.1f} MB')
print(f'Rows={len(train_df):,}  Cols={train_df.shape[1]}')
print('\nDone. Use this CSV for temporal train/test split + XGBoost in Sprint 2.')
print('Do not randomly shuffle rows — split on Month.')


## 5. Sanity checks (anti-leakage)

Confirm the label is not identical to a pure current-month feature copy, and that eligible tenure respects the grace period.


In [ ]:
# Label should differ from current-month "gone" for some rows (otherwise t+1 shift failed).
same_as_current = (train_df['gone_this_month'] == train_df['Is_Churned']).mean()
print(f'share where gone_this_month == Is_Churned: {same_as_current:.2%} (expect << 100%)')

assert train_df['Months_Since_Creation'].min() >= GRACE_MONTHS
assert train_df['Is_Churned'].isin([0, 1]).all()
assert not train_df['Is_Churned'].isna().any()

# Spot-check: no column name looks like a future peek.
banned_substrings = ('t+1', 'next_month', 'future')
for c in train_df.columns:
    low = c.lower()
    assert not any(b in low for b in banned_substrings), c

print('Sanity checks passed.')


In [ ]:
import pandas as pd
df = pd.read_csv(r"..\..\Data\processed\training_data.csv")
print(df.shape)
print(df["Is_Churned"].mean())
print(df.head(3))
print([c for c in df.columns if "t+1" in c.lower() or "next" in c.lower()])

### Empirical Target Leakage Test (Random Forest)

The column-name check above is cosmetic. The true test for mathematical target leakage (e.g. accidentally rolling future amounts into past features) is an empirical one. 
If a single feature is suspiciously powerful (e.g. Gini importance > 0.50), it is almost certainly leaking the `Is_Churned` target from the future.

In [ ]:
# Quick empirical target leakage check
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Drop identifier columns and the target label
drop_cols = ['customer_id', 'Month', 'state', 'Is_Churned', 'gone_this_month']
X_cols = [c for c in train_df.columns if c not in drop_cols]
X = train_df[X_cols].fillna(0) # naive fill for quick check
y = train_df['Is_Churned']

# Subsample for speed (50k rows)
sample_size = min(50000, len(X))
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=sample_size, replace=False)
X_sample = X.iloc[sample_idx]
y_sample = y.iloc[sample_idx]

# Train a fast tree
rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

# Feature importance
importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)

print("Top 10 Feature Importances (Gini):")
print(importances.head(10))

# Warning threshold
if importances.iloc[0] > 0.5:
    print(f"\n⚠️ WARNING: Feature '{importances.index[0]}' has >50% importance. This is highly suspicious and likely indicates Target Leakage!")
else:
    print("\n✅ No single feature overwhelmingly dominates the split. This is a good sign (though not a 100% guarantee) of no leakage.")